# Transformações e Preparação Analítica do Dataset Steam

Este notebook apresenta o processo de transformação dos dados previamente limpos e estruturados, com o objetivo de preparar o dataset para responder às perguntas analíticas definidas no projeto.

Nesta etapa, são realizadas transformações, criação de variáveis derivadas, indicadores e reorganizações necessárias para apoiar as análises posteriores.

## Importar libs

In [2]:
%pip install pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np



## Carregamento da Camada Silver

Realizado o carregamento do dataset silver previamente gerada no notebook de limpeza. Essa versão contém os dados já tratados e preparados para as transformações analíticas necessárias à construção da versão gold.

In [6]:
df = pd.read_parquet(
    'dataset_silver.parquet',
    engine='fastparquet'
)

In [7]:
df.head()

,appid,name,developer,publisher,positive,negative,owners,average_forever,average_2weeks,price
0,1623730,Palworld,Pocketpair,Pocketpair,358266,22443,"50,000,000 .. 100,000,000",3854,835,2999
1,1938090,Call of Duty: Modern Warfare II,"Treyarch, Raven Software, Beenox, High Moon St...",Activision,419594,294520,"50,000,000 .. 100,000,000",5397,639,3849
2,1063730,New World: Aeternum,Amazon Games,Amazon Games,196798,90080,"50,000,000 .. 100,000,000",10588,18,5999
3,2358720,Black Myth: Wukong,Game Science,Game Science,1111720,38378,"50,000,000 .. 100,000,000",3268,524,5999
4,550,Left 4 Dead 2,Valve,Valve,940221,23762,"50,000,000 .. 100,000,000",2479,352,999


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7166 entries, 0 to 7165
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   appid            7166 non-null   int64 
 1   name             7166 non-null   object
 2   developer        7141 non-null   object
 3   publisher        7119 non-null   object
 4   positive         7166 non-null   int64 
 5   negative         7166 non-null   int64 
 6   owners           7166 non-null   object
 7   average_forever  7166 non-null   int64 
 8   average_2weeks   7166 non-null   int64 
 9   price            7166 non-null   int64 
dtypes: int64(6), object(4)
memory usage: 560.0+ KB


## Transformações Analíticas do Dataset

Após o carregamento do dataset previamente limpo e estruturado, serão realizadas transformações nos dados com o objetivo de preparar a base para responder às questões analíticas definidas no projeto.

As transformações incluem criação de variáveis derivadas, conversões de escala, cálculos de indicadores e reorganizações necessárias para apoiar as análises posteriores. Essas novas informações serão armazenadas em novas colunas no dataset, preservando os atributos originais sem alterações diretas.

A manutenção das colunas originais tem como objetivo garantir a rastreabilidade do processo de preparação dos dados, permitindo identificar claramente quais variáveis foram derivadas, transformadas ou calculadas ao longo da construção da versão gold do dataset.

## Criação da Variável `preco_usd`

A variável `price` encontra-se representada em centavos no dataset original. Para adequar os valores às métricas analíticas definidas no projeto, foi criada uma nova variável chamada `preco_usd`, representando o preço do jogo em dólares americanos.

A coluna original `price` foi mantida sem alterações para preservar a rastreabilidade dos dados.

Essa transformação é necessária para o cálculo correto da métrica de custo por hora jogada.

In [9]:
df['preco_usd'] = df['price'] / 100

In [10]:
df[['name', 'price', 'preco_usd']].head()

,name,price,preco_usd
0,Palworld,2999,29.99
1,Call of Duty: Modern Warfare II,3849,38.49
2,New World: Aeternum,5999,59.99
3,Black Myth: Wukong,5999,59.99
4,Left 4 Dead 2,999,9.99


## Criação da Variável `tempo_total_horas`

A coluna original `average_forever` foi mantida sem alterações para preservar a rastreabilidade dos dados. Como essa variável representa o tempo médio total de jogo em minutos, foi criada a variável derivada `tempo_total_horas`, convertendo os valores para horas.

Essa transformação foi realizada para adequar os dados às métricas definidas no problema analítico, além de facilitar a interpretação e comparação do tempo médio de jogo entre os títulos analisados.

In [14]:
df['tempo_total_horas'] = (
    df['average_forever'] / 60
).round(2)

In [15]:
df[['name', 'average_forever', 'tempo_total_horas']].head()

,name,average_forever,tempo_total_horas
0,Palworld,3854,64.23
1,Call of Duty: Modern Warfare II,5397,89.95
2,New World: Aeternum,10588,176.47
3,Black Myth: Wukong,3268,54.47
4,Left 4 Dead 2,2479,41.32


## Criação da Variável `custo_hora`

Foi criada a variável derivada `custo_hora`, responsável por representar quanto o jogador paga, em média, por cada hora de jogo obtida.

O cálculo utiliza o preço do jogo em dólares (`preco_usd`) e o tempo médio total jogado em horas (`tempo_total_horas`). Essa métrica corresponde ao principal indicador de custo-benefício definido no problema analítico do projeto.

In [16]:
df['custo_hora'] = (
    df['preco_usd'] / df['tempo_total_horas']
).round(2)

In [17]:
df[['name', 'preco_usd', 'tempo_total_horas', 'custo_hora']].head()

,name,preco_usd,tempo_total_horas,custo_hora
0,Palworld,29.99,64.23,0.47
1,Call of Duty: Modern Warfare II,38.49,89.95,0.43
2,New World: Aeternum,59.99,176.47,0.34
3,Black Myth: Wukong,59.99,54.47,1.10
4,Left 4 Dead 2,9.99,41.32,0.24


## Criação da Variável `owners_medio`

A variável `owners` apresenta a estimativa de proprietários dos jogos em formato textual de intervalo, como por exemplo `"100,000 .. 200,000"`. Embora essa representação seja adequada para leitura descritiva, ela dificulta operações matemáticas, cálculos estatísticos e comparações analíticas entre os jogos.

Para permitir a utilização dessa informação em métricas e indicadores do projeto, foi criada a variável derivada `owners_medio`, calculada a partir da média entre o limite inferior e o limite superior de cada faixa de proprietários.

Essa transformação permitirá utilizar a estimativa de popularidade dos jogos em análises posteriores, incluindo cálculos de percentis e identificação de títulos classificados como hidden gems.


In [18]:
def calcular_media_owners(faixa):
    limites = faixa.replace(',', '').split(' .. ')
    limite_inferior = int(limites[0])
    limite_superior = int(limites[1])
    return (limite_inferior + limite_superior) / 2

df['owners_medio'] = df['owners'].apply(calcular_media_owners)

O método `calcular_media_owners` recebe o dado da coluna `owners`, remove as vírgulas dos valores numéricos e separa o limite inferior e superior do intervalo. Em seguida, esses limites são convertidos para números inteiros e é calculada a média entre eles.

A função é aplicada a todos os registros da coluna `owners`, gerando a nova variável `owners_medio`, que representa uma estimativa numérica da quantidade de proprietários de cada jogo.

In [20]:
df[['name', 'owners', 'owners_medio']].head()

,name,owners,owners_medio
0,Palworld,"50,000,000 .. 100,000,000",75000000.0
1,Call of Duty: Modern Warfare II,"50,000,000 .. 100,000,000",75000000.0
2,New World: Aeternum,"50,000,000 .. 100,000,000",75000000.0
3,Black Myth: Wukong,"50,000,000 .. 100,000,000",75000000.0
4,Left 4 Dead 2,"50,000,000 .. 100,000,000",75000000.0


## Criação da Variável `steam_rating`

As variáveis `positive` e `negative` representam, respectivamente, a quantidade de avaliações positivas e negativas recebidas pelos jogos na plataforma Steam. Para transformar essas informações em uma métrica comparável de qualidade percebida pelos usuários, foi criada a variável derivada `steam_rating`.

Essa métrica representa a proporção de avaliações positivas em relação ao total de avaliações recebidas pelo jogo, produzindo um indicador padronizado entre 0 e 1. Valores mais próximos de 1 indicam maior aceitação dos usuários.

In [21]:
df['steam_rating'] = (
    df['positive'] /
    (df['positive'] + df['negative'])
).round(4)

In [22]:
df[['name', 'positive', 'negative', 'steam_rating']].head()

,name,positive,negative,steam_rating
0,Palworld,358266,22443,0.9410
1,Call of Duty: Modern Warfare II,419594,294520,0.5876
2,New World: Aeternum,196798,90080,0.6860
3,Black Myth: Wukong,1111720,38378,0.9666
4,Left 4 Dead 2,940221,23762,0.9754
